# 07 - FAISS Vector Index (Local Semantic Search)

**Goal**: Build a local FAISS index over job description embeddings so we can retrieve
similar postings instantly.

**Learning concepts**: cosine similarity, vector indexing, metadata alignment,
artifact caching.

---

In [1]:
%pip install numpy pandas

print(f"Importing libraries and modules...")

FORCE_RECOMPUTE = False  # Set to True to rebuild FAISS index from scratch

import numpy as np
import pandas as pd

from talentlens.config import (
    POSTINGS_NLP_PARQUET,
    EMBEDDINGS_NPY,
    EMBEDDING_JOB_IDS_NPY,
    FAISS_INDEX_PATH,
    RETRIEVAL_META_PARQUET,
)
from talentlens.vector_index import (
    build_faiss_index,
    build_retrieval_meta,
    load_faiss_index,
    save_faiss_index,
    search_faiss,
)

pd.set_option('display.max_columns', None)

print(f"[SUCCESS] Libraries imported successfully.")

Note: you may need to restart the kernel to use updated packages.
Importing libraries and modules...
[SUCCESS] Libraries imported successfully.


## 1. Load embeddings + minimal metadata

We assume embeddings were produced in notebook 06.

- `EMBEDDINGS_NPY`: (N, 384)
- `EMBEDDING_JOB_IDS_NPY`: (N,) job_id alignment for each embedding row
- `POSTINGS_NLP_PARQUET`: source dataframe for metadata (title/location/desc_clean)

Alignment matters: the index row \\(i\\) must map to the same job_id as `job_ids[i]`.

In [2]:
for p in [POSTINGS_NLP_PARQUET, EMBEDDINGS_NPY, EMBEDDING_JOB_IDS_NPY]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required artifact not found: {p}. Run notebooks 04 and 06 first."
        )

df = pd.read_parquet(POSTINGS_NLP_PARQUET)
embeddings = np.load(EMBEDDINGS_NPY)
job_ids = np.load(EMBEDDING_JOB_IDS_NPY)

print(f"df shape: {df.shape}")
print(f"embeddings: {embeddings.shape} dtype={embeddings.dtype}")
print(f"job_ids: {job_ids.shape} dtype={job_ids.dtype}")

if len(df) != len(embeddings) or len(job_ids) != len(embeddings):
    raise ValueError(
        "Artifact alignment mismatch. Ensure notebook 06 saved embeddings for the same "
        "rows in postings_nlp.parquet."
    )

# Sanity check: job_id alignment
same = (df['job_id'].values == job_ids).mean()
print(f"Job ID alignment match rate: {same*100:.2f}%")
if same < 0.99:
    print(
        "[WARNING] job_id alignment is not perfect. We'll trust job_ids.npy for mapping. "
        "(This can happen if the dataframe was re-ordered between saves.)"
    )

df shape: (123842, 42)
embeddings: (123842, 384) dtype=float32
job_ids: (123842,) dtype=int64
Job ID alignment match rate: 100.00%


## 2. Build (or load) the FAISS index

We use cosine similarity by L2-normalizing vectors and using an inner product index.

**Caching**: if `FAISS_INDEX_PATH` exists and `FORCE_RECOMPUTE=False`, we load it.
Otherwise we rebuild and save.

In [3]:
if (not FORCE_RECOMPUTE) and FAISS_INDEX_PATH.exists():
    print(f"Loading FAISS index from {FAISS_INDEX_PATH}...")
    index = load_faiss_index(FAISS_INDEX_PATH)
    print("[SUCCESS] FAISS index loaded.")
else:
    print("Building FAISS index (this is fast for ~100K vectors)...")
    index = build_faiss_index(embeddings, normalize=True)
    save_faiss_index(index, FAISS_INDEX_PATH)
    print(f"[SUCCESS] FAISS index saved to {FAISS_INDEX_PATH}")

print(index)

Building FAISS index (this is fast for ~100K vectors)...
[SUCCESS] FAISS index saved to D:\GigaDocuments\projects\TalentLens\models\faiss_index\postings.index
<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001E874984240> >


## 3. Save compact metadata for retrieval demos

We store a small table aligned to embeddings/index rows so future notebooks don't need to
load the full NLP parquet just to display results.

In [4]:
if (not FORCE_RECOMPUTE) and RETRIEVAL_META_PARQUET.exists():
    print(f"Retrieval meta already exists: {RETRIEVAL_META_PARQUET}")
else:
    meta = build_retrieval_meta(
        job_ids=job_ids,
        titles=df['title'].fillna('').values,
        locations=df['location'].fillna('').values,
        desc_clean=df.get('desc_clean', pd.Series([''] * len(df))).fillna('').values,
    )
    meta_df = pd.DataFrame(meta)
    meta_df.to_parquet(RETRIEVAL_META_PARQUET, index=False)
    print(f"[SUCCESS] Saved retrieval meta to {RETRIEVAL_META_PARQUET}")

[SUCCESS] Saved retrieval meta to D:\GigaDocuments\projects\TalentLens\data\processed\retrieval_meta.parquet


## 4. Demo: search similar jobs

Pick a random job, search the FAISS index, and display the nearest neighbors.

In [5]:
meta_df = pd.read_parquet(RETRIEVAL_META_PARQUET)

rng = np.random.default_rng(42)
query_idx = int(rng.integers(0, len(embeddings)))
query_vec = embeddings[query_idx]

res = search_faiss(index, query_vec, k=6, normalize_query=True)
hits = meta_df.iloc[res.indices].copy()
hits['score'] = res.scores
hits['rank'] = np.arange(len(hits))

print("Query job:")
display(meta_df.iloc[[query_idx]])

print("\nTop neighbors (including self at rank 0):")
display(hits[['rank', 'score', 'job_id', 'title', 'location', 'desc']].head(6))

Query job:


,job_id,title,location,desc
11053,3887476076,Scientist,"Malvern, PA","title: scientist 2 location: malvern, pa durat..."



Top neighbors (including self at rank 0):


,rank,score,job_id,title,location,desc
11053,0,1.000000,3887476076,Scientist,"Malvern, PA","title: scientist 2 location: malvern, pa durat..."
10807,1,0.938970,3887467922,Chemistry Scientist,"Pennsylvania, United States",description: • bachelor’s degree or equivalent...
12709,2,0.921133,3887702242,Scientist (Mammalian Cell Culture/ELISA/Potenc...,"Malvern, PA","duties:bachelor’s degree or equivalent, in bio..."
50491,3,0.710444,3901373815,Biochemist,"Hopewell, NJ",title: qc biochemistry analystposition type: c...
12634,4,0.657666,3887701109,"Associate Scientist, QC","Concord, OH","job title: associate scientist, qc essential r..."
11687,5,0.657666,3887497638,"Associate Scientist, QC","Concord, OH","job title: associate scientist, qc essential r..."
